In [1]:
#| label: fig4

import numpy as np
from PIL import Image
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Load k-space and reconstruct the original image
k_space_orig = np.load('kspace_combined.npy')
img_orig = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(k_space_orig)))

# Function to rotate matrix using PIL (separating real and imaginary parts)
def rotate_matrix(mat, angle):
    pil_img = Image.fromarray(mat.astype(np.float32))
    return np.array(pil_img.rotate(angle), dtype=np.float64)

rows, cols = k_space_orig.shape
split_line = (rows // 2) + 5  # Line where motion occurs (5 lines after the center)[cite: 1]

# 2. Initialize Plotly subplots
fig = make_subplots(rows=1, cols=2, subplot_titles=("Composite k-space (Motion)", "Image Magnitude (Motion Artifact)"))
frames = []
steps = []

# 3. Pre-calculate frames for different rotation angles (e.g., 0° to 30°)
angles = [0, 5, 10, 15, 20, 25, 30]

for i, angle in enumerate(angles):
    if angle == 0:
        k_space_composite = np.copy(k_space_orig)
    else:
        # Apply rotation to the image space data
        img_rot_real = rotate_matrix(np.real(img_orig), angle)
        img_rot_imag = rotate_matrix(np.imag(img_orig), angle)
        img_rot = img_rot_real + 1j * img_rot_imag

        # Transform rotated image back to k-space
        k_space_rot = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(img_rot)))

        # Splice k-spaces to simulate motion[cite: 1]
        k_space_composite = np.zeros_like(k_space_orig, dtype=complex)
        k_space_composite[:split_line, :] = k_space_orig[:split_line, :]
        k_space_composite[split_line:, :] = k_space_rot[split_line:, :]

    # Reconstruct final corrupted image
    img_simulated = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(k_space_composite)))
    img_mag_simulated = np.abs(img_simulated)

    # Logarithmic transformation for k-space visualization[cite: 2]
    k_space_log = np.log(np.abs(k_space_composite) + 1e-5)

    # Add initial traces for Frame 0 (0° rotation)
    if i == 0:
        fig.add_trace(go.Heatmap(z=k_space_log, colorscale='gray', showscale=False), row=1, col=1)
        fig.add_trace(go.Heatmap(z=img_mag_simulated, colorscale='gray', showscale=False), row=1, col=2)

    # Append frame for the current angle
    frames.append(go.Frame(data=[go.Heatmap(z=k_space_log), go.Heatmap(z=img_mag_simulated)], name=str(angle)))

    # Configure slider step
    step = dict(
        method="animate",
        args=[[str(angle)], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}],
        label=f"{angle}°"
    )
    steps.append(step)

# 4. Apply layout, slider, and lock aspect ratio
fig.frames = frames
sliders = [dict(
    active=0,
    currentvalue={"prefix": "Rotation Angle: "},
    pad={"t": 50},
    steps=steps
)]

fig.update_layout(
    sliders=sliders, 
    height=500, 
    title="Interactive Motion Simulation (Sudden Head Rotation)",
    yaxis=dict(scaleanchor="x", autorange="reversed"),
    yaxis2=dict(scaleanchor="x2", autorange="reversed")
)

fig